# LifeMate QLoRA SFT — Kaggle T4 / P100
Upload the repository scripts and data/sft_train_full.jsonl as a private Kaggle Dataset. Enable GPU and Internet for dependency/model download during training. Final inference must be offline. One process uses device_map=auto; two T4 GPUs provide model sharding, not DDP.

In [ ]:
%pip install "transformers>=4.51.3,<5" "peft>=0.15,<1" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" safetensors


In [ ]:
from pathlib import Path
import torch, subprocess, sys
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
root = Path("/kaggle/input")
scripts = list(root.rglob("kaggle_train_sft.py"))
datasets = list(root.rglob("sft_train_full.jsonl"))
assert len(scripts) == len(datasets) == 1, "Attach exactly one training script and one dataset"
print(torch.cuda.device_count(), torch.cuda.get_device_name(0))
subprocess.run([sys.executable, str(scripts[0]), "--data", str(datasets[0]), "--out", "/kaggle/working/out/lora_adapter", "--model", "Qwen/Qwen3-4B-Instruct-2507", "--epochs", "3", "--rank", "32", "--alpha", "64", "--batch_size", "1", "--grad_accum", "16", "--max_len", "4096", "--qlora"], check=True)


In [ ]:
import json
adapter = Path("/kaggle/working/out/lora_adapter/final_adapter")
config = json.loads((adapter / "adapter_config.json").read_text())
assert config["r"] == 32 and config["lora_alpha"] == 64
assert (adapter / "adapter_model.safetensors").stat().st_size > 0
print("Adapter verified:", adapter)
import shutil
shutil.make_archive("/kaggle/working/lifemate_final_adapter", "zip", adapter)
